# Clasificación de severidad de daños vehiculares para triage de siniestros
## CNN desde cero + comparación con Transfer Learning + Auditoría con Grad-CAM

**Autor:** César | Curso Deep Learning | Diplomado Data Analytics ESAN
**Fundamento teórico:** Selvaraju et al. (2020), *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization*, IJCV.

---

### Contextualización del problema de negocio

En el flujo actual de una aseguradora, un ajustador humano revisa fotos del vehículo siniestrado para
estimar la severidad del daño y aproximar el costo de reparación. Este proceso es lento y es el principal
cuello de botella en el tiempo de resolución de un siniestro (`claim cycle time`). Un clasificador automático
de severidad no reemplaza al ajustador, pero permite **triage**: enrutar los casos "minor" a aprobación
rápida/automática, y priorizar la revisión humana en los casos "moderate"/"severe" donde el costo de un
error es más alto.

**Por qué esto no es un problema categórico estándar:** las clases `minor < moderate < severe` tienen
un orden. Confundir `minor` con `severe` es un error mucho más costoso para el negocio que confundir
`minor` con `moderate`. Cross-Entropy estándar no distingue esto — lo vamos a medir explícitamente con
**Cohen's Kappa cuadrático**, que penaliza más los errores "lejanos" en la escala ordinal.

**Por qué Grad-CAM es indispensable aquí, no un accesorio:** un modelo con buen accuracy pero que
aprendió a inferir severidad a partir de correlaciones espurias (ej. el color del auto, el fondo de la foto,
la marca) es un riesgo legal y de negocio inaceptable en un sistema que afecta pagos de seguros. Grad-CAM
es nuestra herramienta de auditoría antes de considerar cualquier despliegue.

### Limitación de validez externa (documentar en el informe)
Este dataset de Kaggle contiene fotos ya curadas y encuadradas hacia el daño. En producción, las fotos
las toma el cliente con su celular: ángulos inconsistentes, fondos variables, iluminación deficiente. El
desempeño reportado aquí es un **techo optimista**, no una estimación de desempeño en producción.


## 1. Setup y dependencias

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn kagglehub -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report, cohen_kappa_score)
from collections import Counter
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = ["minor", "moderate", "severe"]  # orden ordinal explícito -> índices 0,1,2

## 2. Data

Dataset: **Car Damage Severity Dataset** (Kaggle, `prajwalbhamere/car-damage-severity-dataset`).
Carpetas `training/` y `validation/`, cada una con subcarpetas `minor/`, `moderate/`, `severe/`.

> Nota: no existe un split `test/` independiente en el dataset original. Vamos a partir `validation/`
> en val/test estratificado, para no reportar métricas finales sobre el mismo set que usamos para
> early stopping (evitaría un sesgo optimista silencioso).


In [ ]:
import kagglehub

path = kagglehub.dataset_download("prajwalbhamere/car-damage-severity-dataset")
print("Dataset en:", path)
print(os.listdir(path))

In [ ]:
# Estructura real del dataset descargado: {path}/data3a/{training,validation}/{01-minor,02-moderate,03-severe}
# Los prefijos numéricos (01-, 02-, 03-) preservan el orden ordinal minor<moderate<severe al alfabetizar,
# que es como ImageFolder asigna los índices de clase.
TRAIN_DIR = os.path.join(path, "data3a", "training")
VAL_DIR   = os.path.join(path, "data3a", "validation")

for split_dir, split_name in [(TRAIN_DIR, "train"), (VAL_DIR, "validation")]:
    print(f"--- {split_name} ---")
    for cls in sorted(os.listdir(split_dir)):
        p = os.path.join(split_dir, cls)
        if os.path.isdir(p):
            print(f"  {cls}: {len(os.listdir(p))} imágenes")

In [ ]:
from sklearn.model_selection import train_test_split

full_val = datasets.ImageFolder(VAL_DIR)
print("Mapeo de clases:", full_val.class_to_idx)  # debe respetar orden minor=0, moderate=1, severe=2

val_targets = [s[1] for s in full_val.samples]
val_idx, test_idx = train_test_split(
    range(len(full_val)), test_size=0.5, stratify=val_targets, random_state=SEED
)
print(f"Val: {len(val_idx)} | Test: {len(test_idx)}")

## 3. Pipeline de datos

**Dos pipelines distintos y por qué:**
- Para el **CNN desde cero**: normalización simple `(0.5, 0.5, 0.5)` — no hay razón para usar las
  estadísticas de ImageNet si no hay pesos preentrenados de por medio.
- Para el **ResNet18 con transfer learning**: normalización con medias/std de ImageNet — el backbone
  preentrenado espera esa distribución de entrada.

**Augmentation más agresivo que en el proyecto anterior:** al entrenar desde cero no tenemos el prior
de ImageNet actuando como regularizador implícito, así que compensamos con más augmentation
(rotación, flip horizontal, jitter de color, recorte aleatorio) para reducir el riesgo de overfitting
con un dataset de pocos miles de imágenes.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
SIMPLE_MEAN = [0.5, 0.5, 0.5]
SIMPLE_STD = [0.5, 0.5, 0.5]
IMG_SIZE = 224

def make_transforms(mean, std, train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

# Pipelines para el CNN desde cero
train_tf_scratch = make_transforms(SIMPLE_MEAN, SIMPLE_STD, train=True)
eval_tf_scratch  = make_transforms(SIMPLE_MEAN, SIMPLE_STD, train=False)

# Pipelines para ResNet18 (transfer learning)
train_tf_resnet = make_transforms(IMAGENET_MEAN, IMAGENET_STD, train=True)
eval_tf_resnet  = make_transforms(IMAGENET_MEAN, IMAGENET_STD, train=False)

BATCH_SIZE = 32

def build_loaders(train_tf, eval_tf):
    train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
    val_ds_full = datasets.ImageFolder(VAL_DIR, transform=eval_tf)
    val_ds = torch.utils.data.Subset(val_ds_full, val_idx)
    test_ds = torch.utils.data.Subset(val_ds_full, test_idx)

    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2),
        DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
        DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2),
        train_ds,
    )

train_loader_s, val_loader_s, test_loader_s, train_ds_s = build_loaders(train_tf_scratch, eval_tf_scratch)
train_loader_r, val_loader_r, test_loader_r, train_ds_r = build_loaders(train_tf_resnet, eval_tf_resnet)

print(f"Train: {len(train_ds_s)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

In [ ]:
# Class weights (el dataset típicamente trae más ejemplos de 'severe' que de 'minor')
train_targets = [s[1] for s in train_ds_s.samples]
class_counts = Counter(train_targets)
total = sum(class_counts.values())
n_classes = len(CLASS_NAMES)

class_weights = torch.tensor(
    [total / (n_classes * class_counts[i]) for i in range(n_classes)],
    dtype=torch.float32
).to(device)

print("Distribución de clases (train):", {CLASS_NAMES[k]: v for k, v in class_counts.items()})
print("Class weights:", class_weights.tolist())

## 4. Arquitectura A — CNN desde cero

**Decisión de diseño deliberada:** en lugar de terminar con `Flatten() + Dense`, usamos
**Global Average Pooling (GAP)** justo antes de la capa final. Esto no es solo una elección de
regularización (GAP tiene cero parámetros vs. una Dense de miles) — es la arquitectura que el propio
paper de Grad-CAM identifica como **caso especial exacto de CAM** (Sec. 3.1, Ec. 3-11): cuando el mapa
de features pasa directo por GAP a una capa lineal, los pesos de esa capa lineal son *matemáticamente
equivalentes* a los pesos de importancia que Grad-CAM calcula vía gradientes. Es la arquitectura más
"honesta" para razonar sobre qué está mirando el modelo.

**Justificación de hiperparámetros:**

| Elemento | Valor | Justificación |
|---|---|---|
| Bloques conv | 4 (32→64→128→256 canales) | Duplicar canales al reducir resolución espacial es el patrón estándar para preservar capacidad representacional |
| BatchNorm | después de cada conv | Sin pesos preentrenados, BN es crítico para estabilizar el entrenamiento desde una inicialización aleatoria |
| Dropout | 0.5 antes de la capa final | Único punto de regularización fuerte, dado que no hay prior de ImageNet |
| GAP en vez de Flatten | — | Menos parámetros, menos overfitting, y compatibilidad directa con CAM/Grad-CAM |


In [ ]:
class CarDamageCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(
            conv_block(3, 32),     # 224 -> 112
            conv_block(32, 64),    # 112 -> 56
            conv_block(64, 128),   # 56 -> 28
            conv_block(128, 256),  # 28 -> 14   <- última capa conv, aquí enganchamos Grad-CAM
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

model_scratch = CarDamageCNN(num_classes=3).to(device)

total_params = sum(p.numel() for p in model_scratch.parameters())
print(f"Parámetros totales (CNN desde cero): {total_params:,}")

## 5. Arquitectura B — ResNet18 con Transfer Learning (baseline de comparación)

Mismo criterio de congelamiento que en el proyecto anterior: `conv1` a `layer2` congeladas (features
genéricas), `layer3`/`layer4` descongeladas (adaptación semántica al dominio), `fc` reemplazada.


In [ ]:
def build_resnet_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for name, param in m.named_parameters():
        param.requires_grad = name.startswith("layer3") or name.startswith("layer4") or name.startswith("fc")
    m.fc = nn.Linear(m.fc.in_features, 3)
    return m.to(device)

model_resnet = build_resnet_model()
trainable = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_resnet.parameters())
print(f"ResNet18 -- Parámetros entrenables: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 6. Entrenamiento (función genérica para ambos modelos)

**Diferencia clave de hiperparámetros entre A y B:**
- CNN desde cero: LR más alto (1e-3), más épocas (patience mayor) — parte de pesos aleatorios, necesita
  más señal de gradiente y más tiempo para converger.
- ResNet18: LR bajo con esquema diferencial (ya usado en el proyecto anterior) — los pesos ya están
  cerca de un buen óptimo, solo se ajustan finamente.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(y.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(labels_all, preds_all)
    return avg_loss, acc


def train_model(model, train_loader, val_loader, optimizer, criterion, scheduler,
                 epochs, patience, tag):
    best_val_loss, patience_counter = float("inf"), 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    ckpt_path = f"best_{tag}.pt"

    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step(val_loss)

        history["train_loss"].append(tr_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc); history["val_acc"].append(val_acc)

        print(f"[{tag}] Epoch {epoch+1}/{epochs} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
              f"val_loss={val_loss:.4f} acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss, patience_counter = val_loss, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[{tag}] Early stopping en epoch {epoch+1}")
                break

    model.load_state_dict(torch.load(ckpt_path))
    return history

In [ ]:
# --- Entrenamiento CNN desde cero ---
criterion_s = nn.CrossEntropyLoss(weight=class_weights)
optimizer_s = torch.optim.AdamW(model_scratch.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_s = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_s, mode="min", factor=0.5, patience=2)

history_scratch = train_model(
    model_scratch, train_loader_s, val_loader_s, optimizer_s, criterion_s, scheduler_s,
    epochs=40, patience=6, tag="scratch"
)

In [ ]:
# --- Entrenamiento ResNet18 (transfer learning) ---
criterion_r = nn.CrossEntropyLoss(weight=class_weights)
optimizer_r = torch.optim.AdamW([
    {"params": model_resnet.layer3.parameters(), "lr": 1e-4},
    {"params": model_resnet.layer4.parameters(), "lr": 1e-4},
    {"params": model_resnet.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-4)
scheduler_r = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_r, mode="min", factor=0.5, patience=1)

history_resnet = train_model(
    model_resnet, train_loader_r, val_loader_r, optimizer_r, criterion_r, scheduler_r,
    epochs=15, patience=3, tag="resnet"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_scratch["val_loss"], label="CNN desde cero")
axes[0].plot(history_resnet["val_loss"], label="ResNet18 (TL)")
axes[0].set_title("Val Loss"); axes[0].legend()
axes[1].plot(history_scratch["val_acc"], label="CNN desde cero")
axes[1].plot(history_resnet["val_acc"], label="ResNet18 (TL)")
axes[1].set_title("Val Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Evaluación comparativa en test set

**Por qué Cohen's Kappa cuadrático además de accuracy/F1:**
Penaliza más los errores "lejanos" en la escala ordinal (confundir minor↔severe pesa más que
minor↔moderate). Un modelo con el mismo accuracy pero mayor Kappa cuadrático está cometiendo errores
"más baratos" para el negocio — esta es la métrica que deberías reportar como criterio principal de
decisión, no accuracy.


In [ ]:
def evaluate_model(model, loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(y.numpy())
    return np.array(labels_all), np.array(preds_all)

y_true_s, y_pred_s = evaluate_model(model_scratch, test_loader_s)
y_true_r, y_pred_r = evaluate_model(model_resnet, test_loader_r)

for name, y_true, y_pred in [("CNN desde cero", y_true_s, y_pred_s), ("ResNet18 (TL)", y_true_r, y_pred_r)]:
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    print(f"Cohen's Kappa (cuadrático): {kappa:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, name, y_true, y_pred in [
    (axes[0], "CNN desde cero", y_true_s, y_pred_s),
    (axes[1], "ResNet18 (TL)", y_true_r, y_pred_r),
]:
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(name); ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.tight_layout(); plt.show()

# Nota: en la matriz de confusión, presta especial atención a las celdas de las esquinas
# (minor predicho como severe, o viceversa) -- esos son los errores ordinales "caros" para el negocio

## 8. Grad-CAM — implementación desde cero (reutilizable para ambos modelos)

Misma implementación matemática de siempre (Selvaraju et al. 2020, Ec. 1-2), pero ahora la vamos a
usar sobre **dos arquitecturas distintas** apuntando cada una a su propia última capa convolucional.


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        logits = self.model(input_tensor)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        self.model.zero_grad()
        score = logits[:, class_idx]
        score.backward(retain_graph=True)

        alpha = self.gradients.mean(dim=(2, 3), keepdim=True)          # Ec. 1: GAP de gradientes
        weighted = (alpha * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(weighted)                                        # Ec. 2: combinación lineal + ReLU

        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, F.softmax(logits, dim=1).detach().cpu().numpy()[0]

# Hook en la última capa conv de cada arquitectura
gradcam_scratch = GradCAM(model_scratch, target_layer=model_scratch.features[-1][0])
gradcam_resnet  = GradCAM(model_resnet, target_layer=model_resnet.layer4[-1])

In [ ]:
def denormalize(tensor, mean, std):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    img = tensor.cpu() * std_t + mean_t
    return img.permute(1, 2, 0).clamp(0, 1).numpy()

def plot_gradcam(model, dataset, idx, grad_cam, mean, std, class_names=CLASS_NAMES, model_name=""):
    img_tensor, true_label = dataset[idx]
    input_tensor = img_tensor.unsqueeze(0).to(device)
    cam, pred_class, probs = grad_cam.generate(input_tensor)
    img = denormalize(img_tensor, mean, std)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(model_name)
    axes[0].imshow(img); axes[0].set_title(f"Original\nReal: {class_names[true_label]}")
    axes[1].imshow(cam, cmap="jet"); axes[1].set_title("Grad-CAM raw")
    axes[2].imshow(img); axes[2].imshow(cam, cmap="jet", alpha=0.5)
    axes[2].set_title(f"Overlay\nPred: {class_names[pred_class]} ({probs[pred_class]:.2%})")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

# test_ds_scratch y test_ds_resnet comparten los mismos índices de test_idx (misma imagen,
# distinto pipeline de transform) -- por eso un mismo `idx` es comparable entre ambos modelos.
test_ds_scratch = test_loader_s.dataset
test_ds_resnet = test_loader_r.dataset

N_CORRECT_CASES = 5  # <-- cambia este número para ver más o menos casos correctos

# Casos donde AMBOS modelos acertaron, para comparar de forma justa la calidad del Grad-CAM
correct_both_idx = np.where((y_true_s == y_pred_s) & (y_true_r == y_pred_r))[0]
print(f"Casos correctos en ambos modelos: {len(correct_both_idx)} / {len(y_true_s)}")

for idx in correct_both_idx[:N_CORRECT_CASES]:
    plot_gradcam(model_scratch, test_ds_scratch, idx, gradcam_scratch,
                 SIMPLE_MEAN, SIMPLE_STD, model_name="CNN desde cero")
    plot_gradcam(model_resnet, test_ds_resnet, idx, gradcam_resnet,
                 IMAGENET_MEAN, IMAGENET_STD, model_name="ResNet18 (Transfer Learning)")

## 9. Auditoría de negocio: ¿el modelo mira el daño real?

Este es el entregable más importante para una aseguradora. No basta con "el modelo tiene 85% de
accuracy" — hay que demostrar **qué está mirando** para llegar a esa conclusión, antes de siquiera
considerar integrarlo en un flujo que afecta pagos de siniestros.

**Checklist de auditoría a documentar en el informe (con evidencia visual de Grad-CAM):**

1. **¿El heatmap se concentra en la zona de daño (abolladura, rayón, cristal roto)?** Si sí → señal de
   que el modelo aprendió una representación útil.
2. **¿El heatmap se concentra en el fondo, el logo de la marca, o la placa?** Si sí → el modelo está
   usando correlaciones espurias del dataset (ej. cierta marca aparece más en fotos de daño severo por
   sesgo de muestreo) — **no debería desplegarse** sin corregir el dataset.
3. **En los casos mal clasificados, ¿el error es "razonable"?** — si el heatmap muestra que el modelo
   miraba una zona de daño real pero subestimó la severidad (ej. no detectó que el golpe también afectó
   el marco de la puerta), es un error de "grado", más aceptable para el negocio que un error de "mirar
   el lugar equivocado".
4. **Comparación entre modelos:** si ResNet18 (TL) y el CNN desde cero coinciden en la predicción pero
   sus heatmaps difieren en calidad/foco, replica el estudio de "Evaluando confianza" del paper original
   (Sec. 5.2): ¿cuál modelo tiene un Grad-CAM más nítido y localizado? Ese es el modelo más confiable,
   incluso si el accuracy es similar.


In [ ]:
N_MISCLASSIFIED_CASES = 5  # <-- cambia este número para ver más o menos casos mal clasificados

misclassified_idx = np.where(y_true_s != y_pred_s)[0]
print(f"CNN desde cero -- casos mal clasificados: {len(misclassified_idx)} / {len(y_true_s)}")

for idx in misclassified_idx[:N_MISCLASSIFIED_CASES]:
    plot_gradcam(model_scratch, test_ds_scratch, idx, gradcam_scratch,
                 SIMPLE_MEAN, SIMPLE_STD, model_name="CNN desde cero -- caso mal clasificado")
    plot_gradcam(model_resnet, test_ds_resnet, idx, gradcam_resnet,
                 IMAGENET_MEAN, IMAGENET_STD, model_name="ResNet18 (mismo caso)")

## 10. Conclusiones y limitaciones (a expandir en el informe)

- **Comparación cuantitativa:** reportar accuracy, F1 por clase y Kappa cuadrático de ambos modelos en
  una tabla — el criterio de decisión de negocio debería ser Kappa, no accuracy.
- **Auditoría cualitativa:** con base en la Sección 9, concluir si alguno de los dos modelos es
  "desplegable" en su estado actual o si requiere más datos/curación antes de integrarse a un flujo real.
- **Limitación de dataset:** fotos curadas vs. fotos reales de clientes (ver introducción).
- **Limitación de la formulación del problema:** este modelo clasifica severidad, no *localiza* ni
  *cuantifica* el daño (no da un costo estimado de reparación) — es un primer filtro de triage, no un
  reemplazo del ajustador.
- **Extensión futura:** tratar el problema como regresión ordinal (ej. CORAL loss) en vez de clasificación
  categórica, que respeta nativamente el orden minor<moderate<severe.
